In [ ]:
import pandas as pd
import os

ruta_archivo_excel = 'data/combined_dataframe_v4.xlsx'
hojas = ['Dennis', 'Ayrton', 'Ruddy', 'Ricardo', 'Jose']

df_list = []

for hoja in hojas:
    #df_temp = pd.read_excel(ruta_archivo_excel, sheet_name=hoja)
    df_temp = pd.read_excel(ruta_archivo_excel, sheet_name=hoja, engine='openpyxl')

    df_list.append(df_temp)

df = pd.concat(df_list, ignore_index=True)

print("Excel file loaded successfully with added columns 'es,' and 'qu'.")

# Aplicar codificación UTF-8 a las columnas de texto
for col in df.select_dtypes(include=[object]):
    df[col] = df[col].apply(lambda x: x.encode('utf-8').decode('utf-8') if isinstance(x, str) else x)

# Eliminar filas con cualquier valor NaN
df = df.dropna(how='any')

# Mostrar las primeras filas con las dos nuevas columnas
print(df[['es', 'qu']].head())




Excel file loaded successfully with added columns 'es,' and 'qu'.
                                                  es  \
0  Y por último dice que el pueblo de Dios “escap...   
1  ¿Cómo lograban mantenerse estos siervos de tie...   
2  Cabe señalar que solo la Biblia explica de man...   
3  Pero al mismo tiempo nos advierte que “el amor...   
4  Hoy, todo el que desea vivir de acuerdo con su...   

                                                  qu  
0  Ichaqa chaywanpas chay tiempopim llaqtamasikik...  
1  ¿Imaynatam tukuy tiemponkuwan Diosta serviq pu...  
2  Bibliallam allintapuni willawanchik imanasqam ...  
3  Ichaqa nintaqmi: “Qollqella kuyayqa tukuy mana...  
4  Kay tiempopipas Cristopa kamachisqanman hina k...  


# limpiando Outliers

In [ ]:
import pandas as pd

# Elimina valores nulos en las columnas relevantes
df = df.dropna(subset=["es", "qu"])

# Convierte contenido a string para evitar errores al aplicar len()
df["len_es"] = df["es"].astype(str).apply(len)
df["len_qu"] = df["qu"].astype(str).apply(len)
df["abs_diff"] = (df["len_es"] - df["len_qu"]).abs()

# Define la condición original
condicion_original = (
    (df["abs_diff"] > 150) |
    (
        ((df["len_es"] <= 40) | (df["len_qu"] <= 40)) &
        (df["abs_diff"] > 40)
    )
)

# Negar la condición
filtered_df = df[~condicion_original]

# Mostrar solo columnas originales
filtered_df = filtered_df[["es", "qu"]]

print(filtered_df)

                                                       es  \
0       Y por último dice que el pueblo de Dios “escap...   
1       ¿Cómo lograban mantenerse estos siervos de tie...   
2       Cabe señalar que solo la Biblia explica de man...   
3       Pero al mismo tiempo nos advierte que “el amor...   
4       Hoy, todo el que desea vivir de acuerdo con su...   
...                                                   ...   
127069  Por eso se mantienen separados de los utensili...   
127070  Aunque desafió sin temor a 450 profetas de Baa...   
127071          La congregación me ha dado todo su apoyo.   
127072                                ¿Y cómo reaccionan?   
127073                                             Repaso   

                                                       qu  
0       Ichaqa chaywanpas chay tiempopim llaqtamasikik...  
1       ¿Imaynatam tukuy tiemponkuwan Diosta serviq pu...  
2       Bibliallam allintapuni willawanchik imanasqam ...  
3       Ichaqa nintaqmi: “Q

In [ ]:
df=filtered_df

In [ ]:
import re
import unicodedata

def limpiar_texto_avanzado(texto):
    if pd.isnull(texto):
        return ''
    
    # Normalización unicode (quita acentos si lo deseas)
    texto = unicodedata.normalize('NFKC', texto)

    # Eliminar puntuación excepto guiones y apóstrofes relevantes en quechua
    texto = re.sub(r'[^\w\s\'-]', '', texto)

    # Unificar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto)

    # Eliminar espacios al inicio y final
    texto = texto.strip()

    # Convertir a minúsculas
    texto = texto.lower()

    return texto

In [ ]:
print(limpiar_texto_avanzado("asd © - () {}[a].. 1234 PËÉRRO. "))

asd - a 1234 pëérro


In [ ]:
df['es'] = df['es'].astype(str).apply(limpiar_texto_avanzado)
df['qu'] = df['qu'].astype(str).apply(limpiar_texto_avanzado)

In [ ]:
import pandas as pd
import re

def limpiar_texto(texto):
    if pd.isnull(texto):
        return ""
    # Se permiten:
    # Letras con tildes/diéresis/ñ, números, espacios,
    # puntuación común, paréntesis (), corchetes [] y llaves {}
    return re.sub(r"[^a-zA-ZáéíóúüñÁÉÍÓÚÜÑ0-9\s.,!?¿¡:;\"'\-()[\]{}]", "", texto)

df['es'] = df['es'].apply(limpiar_texto)
df['qu'] = df['qu'].apply(limpiar_texto)

In [ ]:
df = df.dropna(subset=['es', 'qu'], how='any')
print(df[['es', 'qu']].head())

                                                  es  \
0  y por último dice que el pueblo de dios escapa...   
1  cómo lograban mantenerse estos siervos de tiem...   
2  cabe señalar que solo la biblia explica de man...   
3  pero al mismo tiempo nos advierte que el amor ...   
4  hoy todo el que desea vivir de acuerdo con sus...   

                                                  qu  
0  ichaqa chaywanpas chay tiempopim llaqtamasikik...  
1  imaynatam tukuy tiemponkuwan diosta serviq pun...  
2  bibliallam allintapuni willawanchik imanasqam ...  
3  ichaqa nintaqmi qollqella kuyayqa tukuy mana a...  
4  kay tiempopipas cristopa kamachisqanman hina k...  


In [ ]:
df.describe()


,es,qu
count,126095,126095
unique,117778,117382
top,por qué,imanasqa
freq,333,392


In [ ]:
#spanish_array = df['es'].tolist()
#quechua_array = df['qu'].tolist()
#source_language_array = df['es'].tolist()
#targe_language_array = df['qu'].tolist()
#name_source_lang = 'Spanish'
#name_target_lang = 'Quechua'

source_language_array = df['es'].tolist()
targe_language_array = df['qu'].tolist()
name_source_lang = 'Spanish'
name_target_lang = 'Quechua'

print(f"{name_source_lang} Array:")
print(source_language_array[:5])  # Mostrar los primeros 5 elementos del array

print(f"{name_target_lang} Array:")
print(targe_language_array[:5])  # Mostrar los primeros 100 caracteres del JSON

Spanish Array:
['y por último dice que el pueblo de dios escapará todo el que se halle escrito en el libro', 'cómo lograban mantenerse estos siervos de tiempo completo', 'cabe señalar que solo la biblia explica de manera satisfactoria cuál es el origen de los muchos idiomas que conocemos hoy', 'pero al mismo tiempo nos advierte que el amor al dinero es raíz de toda clase de males', 'hoy todo el que desea vivir de acuerdo con sus mandamientos considera que conmemorar el aniversario de la muerte de cristo es de suma importancia']
Quechua Array:
['ichaqa chaywanpas chay tiempopim llaqtamasikikuna librasqa kanqaku libropi qillqasqa sutiyuqkunam lliw librakunqaku nispa', 'imaynatam tukuy tiemponkuwan diosta serviq punta cristianokunaqa mantienekuqku', 'bibliallam allintapuni willawanchik imanasqam runakunaqa kunan tiempopi achka rimayniyoq kasqankuta', 'ichaqa nintaqmi qollqella kuyayqa tukuy mana allin ruraykunapa mamanmi nispa', 'kay tiempopipas cristopa kamachisqanman hina kawsaqkunaqa a

In [ ]:

START_TOKEN = '<START>'
PADDING_TOKEN = '<PADDING>'
END_TOKEN = '<END>'
def sort_key(char):
        if char.isdigit():
            return (0, char)  # Numbers first
        elif char.isalpha():
            return (1, char.lower())  # Letters next, case insensitive
        else:
            return (2, char)

def extract_unique_tokens(text_list):
  """Extracts unique tokens (characters) from a list of texts."""
  all_tokens = set()
  for text in text_list:
    for char in text:
      all_tokens.add(char)
  tokens= list(all_tokens)
  tokens.sort()
  #return tokens
  return sorted(tokens, key=sort_key)


source_language_tokens = extract_unique_tokens(source_language_array)
target_language_tokens = extract_unique_tokens(targe_language_array)

source_language_tokens.insert(0,'Ll')
source_language_tokens.insert(0,'Ch')
source_language_tokens.insert(0,'ll')
source_language_tokens.insert(0,'ch')

target_language_tokens.insert(0, 'Ll')
target_language_tokens.insert(0, 'Ch')
target_language_tokens.insert(0, 'll')
target_language_tokens.insert(0, 'ch')

source_language_tokens.insert(0,START_TOKEN)
target_language_tokens.insert(0, START_TOKEN)


source_language_tokens.append(PADDING_TOKEN)
source_language_tokens.append(END_TOKEN)

target_language_tokens.append(PADDING_TOKEN)
target_language_tokens.append(END_TOKEN)

print(f"{name_source_lang} Tokens:")
print(source_language_tokens)

print(f"\n{name_target_lang} Tokens:")
print(target_language_tokens)

source_language_vocabulary = source_language_tokens
target_language_vocabulary = target_language_tokens

print(f"{name_source_lang} Sentences:")
source_language_sentences = source_language_array

print(f"{name_target_lang} Sentences")
target_language_sentences = targe_language_array

print(source_language_sentences[:5])
print(target_language_sentences[:5])

Spanish Tokens:
['<START>', 'ch', 'll', 'Ch', 'Ll', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'é', 'í', 'ñ', 'ó', 'ú', 'ü', ' ', "'", '-', '<PADDING>', '<END>']

Quechua Tokens:
['<START>', 'ch', 'll', 'Ch', 'Ll', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'é', 'í', 'ñ', 'ó', 'ú', 'ü', ' ', "'", '-', '<PADDING>', '<END>']
Spanish Sentences:
Quechua Sentences
['y por último dice que el pueblo de dios escapará todo el que se halle escrito en el libro', 'cómo lograban mantenerse estos siervos de tiempo completo', 'cabe señalar que solo la biblia explica de manera satisfactoria cuál es el origen de los muchos idiomas que conocemos hoy', 'pero al mismo tiempo nos advierte que el amor al dinero es raíz de toda c

In [ ]:
print(len(source_language_sentences))
print(len(target_language_sentences))

126095
126095


In [ ]:
from transformer import Transformer # this is the transformer.py file
import torch
import numpy as np

OSError: [WinError 1114] Error en una rutina de inicialización de biblioteca de vínculos dinámicos (DLL). Error loading "c:\Users\javie\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [54]:
index_to_target = {k:v for k,v in enumerate(target_language_vocabulary)}
target_to_index = {v:k for k,v in enumerate(target_language_vocabulary)}
index_to_source = {k:v for k,v in enumerate(source_language_vocabulary)}
source_to_index = {v:k for k,v in enumerate(source_language_vocabulary)}

In [55]:
source_language_sentences[:10]

['y por último dice que el pueblo de dios escapará todo el que se halle escrito en el libro',
 'cómo lograban mantenerse estos siervos de tiempo completo',
 'cabe señalar que solo la biblia explica de manera satisfactoria cuál es el origen de los muchos idiomas que conocemos hoy',
 'pero al mismo tiempo nos advierte que el amor al dinero es raíz de toda clase de males',
 'hoy todo el que desea vivir de acuerdo con sus mandamientos considera que conmemorar el aniversario de la muerte de cristo es de suma importancia',
 'respuesta el reino de dios es un gobierno celestial y su rey es jesús',
 'cuál es la solución',
 'adán y eva desobedecieron a dios de modo que fueron expulsados del edén',
 'sin embargo poco después de escribir esa carta los de la casa de cloe le informaron de que en la congregación de corinto había graves divisiones',
 'jesús cumplió de forma sorprendente esta profecía durante su ministerio']

In [56]:
target_language_sentences[:10]

['ichaqa chaywanpas chay tiempopim llaqtamasikikuna librasqa kanqaku libropi qillqasqa sutiyuqkunam lliw librakunqaku nispa',
 'imaynatam tukuy tiemponkuwan diosta serviq punta cristianokunaqa mantienekuqku',
 'bibliallam allintapuni willawanchik imanasqam runakunaqa kunan tiempopi achka rimayniyoq kasqankuta',
 'ichaqa nintaqmi qollqella kuyayqa tukuy mana allin ruraykunapa mamanmi nispa',
 'kay tiempopipas cristopa kamachisqanman hina kawsaqkunaqa ancha valorniyoqtam qawanku paypa wañukusqan punchaw yuyariytaqa',
 'kutichiynin hanaq pachapi kaq huk gobiernom kamachiqninñataqmi jesus',
 'imatam rurachwan',
 'adanwan evaqa jehová diostam mana kasukurqakuchu hinaspam edenmanta qarqochikurqaku',
 'ichaqa chay carta qellqasqan qepamanmi cloepa familian pabloman willaykurqaku corinto congregacionpi llumpay liryanakuy kasqanmanta',
 'jesusqa isaiaspa nisqantam kay pachapi diospa munayninta ruraspan allinta cumplirqa']

In [57]:
import numpy as np
PERCENTILE = 97
print( f"{PERCENTILE}th percentile length {name_source_lang}: {np.percentile([len(x) for x in target_language_sentences], PERCENTILE)}" )
print( f"{PERCENTILE}th percentile length {name_target_lang}: {np.percentile([len(x) for x in source_language_sentences], PERCENTILE)}" )


97th percentile length Spanish: 188.0
97th percentile length Quechua: 192.0


In [58]:
max_sequence_length = 200

def is_valid_tokens(sentence, vocab):
    for token in list(set(sentence)):
        if token not in vocab:
            return False
    return True

def is_valid_length(sentence, max_sequence_length):
    return len(list(sentence)) < (max_sequence_length - 1) # need to re-add the end token so leaving 1 space

valid_sentence_indicies = []
for index in range(len(target_language_sentences)):
    kannada_sentence, english_sentence = target_language_sentences[index], source_language_sentences[index]
    if is_valid_length(kannada_sentence, max_sequence_length) \
      and is_valid_length(english_sentence, max_sequence_length) \
      and is_valid_tokens(kannada_sentence, target_language_vocabulary):
        valid_sentence_indicies.append(index)

print(f"Number of sentences: {len(target_language_sentences)}")
print(f"Number of valid sentences: {len(valid_sentence_indicies)}")

Number of sentences: 126095
Number of valid sentences: 121748


In [59]:
target_language_sentences = [target_language_sentences[i] for i in valid_sentence_indicies]
source_language_sentences = [source_language_sentences[i] for i in valid_sentence_indicies]

In [60]:
print(len(source_language_sentences))
print(len(target_language_sentences))

121748
121748


In [61]:
target_language_sentences[:3]

['ichaqa chaywanpas chay tiempopim llaqtamasikikuna librasqa kanqaku libropi qillqasqa sutiyuqkunam lliw librakunqaku nispa',
 'imaynatam tukuy tiemponkuwan diosta serviq punta cristianokunaqa mantienekuqku',
 'bibliallam allintapuni willawanchik imanasqam runakunaqa kunan tiempopi achka rimayniyoq kasqankuta']

In [62]:
import torch

d_model = 512 #
batch_size = 30 #
ffn_hidden = 2048 #
num_heads = 8 #
drop_prob = 0.1 #
num_layers = 3 #
max_sequence_length = 200 #
kn_vocab_size = len(target_language_vocabulary)

transformer = Transformer(d_model,
                          ffn_hidden,
                          num_heads,
                          drop_prob,
                          num_layers,
                          max_sequence_length,
                          kn_vocab_size,
                          source_to_index,
                          target_to_index,
                          START_TOKEN,
                          END_TOKEN,
                          PADDING_TOKEN)

In [63]:
transformer

Transformer(
  (encoder): Encoder(
    (sentence_embedding): SentenceEmbedding(
      (embedding): Embedding(53, 512)
      (position_encoder): PositionalEncoding()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): SequentialEncoder(
      (0): EncoderLayer(
        (attention): MultiHeadAttention(
          (qkv_layer): Linear(in_features=512, out_features=1536, bias=True)
          (linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): LayerNormalization()
        (dropout1): Dropout(p=0.1, inplace=False)
        (ffn): PositionwiseFeedForward(
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (relu): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm2): LayerNormalization()
        (dropout2): Dropout(p=0.1, inplace=False)
      )
      (1): EncoderLayer(
        (attention): MultiHeadAt

In [64]:

#device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [65]:
transformer =transformer.to(device)

In [66]:
NEG_INFTY = -1e9

def create_masks(eng_batch, kn_batch):
    num_sentences = len(eng_batch)
    look_ahead_mask = torch.full([max_sequence_length, max_sequence_length] , True, device=device)
    look_ahead_mask = torch.triu(look_ahead_mask, diagonal=1)
    encoder_padding_mask = torch.full([num_sentences, max_sequence_length, max_sequence_length] , False, device=device)
    decoder_padding_mask_self_attention = torch.full([num_sentences, max_sequence_length, max_sequence_length] , False, device=device)
    decoder_padding_mask_cross_attention = torch.full([num_sentences, max_sequence_length, max_sequence_length] , False, device=device)

    for idx in range(num_sentences):
      eng_sentence_length, kn_sentence_length = len(eng_batch[idx]), len(kn_batch[idx])
      eng_chars_to_padding_mask = np.arange(eng_sentence_length + 1, max_sequence_length)
      kn_chars_to_padding_mask = np.arange(kn_sentence_length + 1, max_sequence_length)
      encoder_padding_mask[idx, :, eng_chars_to_padding_mask] = True
      encoder_padding_mask[idx, eng_chars_to_padding_mask, :] = True
      decoder_padding_mask_self_attention[idx, :, kn_chars_to_padding_mask] = True
      decoder_padding_mask_self_attention[idx, kn_chars_to_padding_mask, :] = True
      decoder_padding_mask_cross_attention[idx, :, eng_chars_to_padding_mask] = True
      decoder_padding_mask_cross_attention[idx, kn_chars_to_padding_mask, :] = True

    encoder_self_attention_mask = torch.where(encoder_padding_mask, NEG_INFTY, 0)
    decoder_self_attention_mask =  torch.where(look_ahead_mask + decoder_padding_mask_self_attention, NEG_INFTY, 0)
    decoder_cross_attention_mask = torch.where(decoder_padding_mask_cross_attention, NEG_INFTY, 0)
    return encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask

## Inference

In [67]:
transformer.eval()
#transformer.load_state_dict(torch.load('transformer_model_3_capas_50epochs.pth', map_location=device))
transformer.load_state_dict(torch.load('transformer_model_3_capas_100epochs_es_qu.pth', map_location=device))

def translate(eng_sentence):
  eng_sentence = (eng_sentence,)
  kn_sentence = ("",)
  for word_counter in range(max_sequence_length):
    encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask= create_masks(eng_sentence, kn_sentence)
    predictions = transformer(eng_sentence,
                              kn_sentence,
                              encoder_self_attention_mask.to(device),
                              decoder_self_attention_mask.to(device),
                              decoder_cross_attention_mask.to(device),
                              enc_start_token=False,
                              enc_end_token=False,
                              dec_start_token=True,
                              dec_end_token=False)
    next_token_prob_distribution = predictions[0][word_counter]
    next_token_index = torch.argmax(next_token_prob_distribution).item()
    next_token = index_to_target[next_token_index]
    kn_sentence = (kn_sentence[0] + next_token, )
    if next_token == END_TOKEN:
      break
  return kn_sentence[0]

In [71]:
translation = translate(limpiar_texto_avanzado("el frio esta insoportable"))
print(translation)
# Hebreos cristianokunaman qellqaspanmi Pablo qawachirqa wakin cristianokuna mana kasukusqankuta, wakin cristianokunaqa manam Diospa munaynin rurakunanpaq (leey Mateo 24: 1 - 4).<END>

lliw allin yachachiqmi kanantaqa<END>


In [69]:
translation = translate("la comida peruana es la más rica del mundo")
print(translation)
#Kay pachapi mana allin ruraykunamanta<END>

kay pachapi mikuymantaqa mikuymi aswan mikun<END>


In [72]:
translation = translate("los pollos estaban piojosos")
print(translation)
#Chiwchikunaqa chay willakuykunam karqa.<END>

chiwchikunaqa usasapam kachkas qaku<END>


In [73]:
translation = translate("son cosas que tengo muy grabadas como si hubieran sucedido ayer")
print(translation)
#Mana allin ruraqkunaqa chaynataq rey kasqanraykum paykunapas karqa.<END>

qawasqaykichikman hina kawsakuspaykichikqa sasachakuypi tarikunkichikmi<END>


In [74]:
translation = translate("no es cierto que los siervos de dios de tiempos modernos estamos llevando a cabo la obra de evangelización más grande de la historia")
print(translation)

kunan tiempopim diosta serviqkunaqa kay tiempopim astawan predicachkanchik aswan vengakuyta atipachkanchik<END>


In [75]:
translation = translate("en esto se mostró el amor de dios para con nosotros en que dios envió a su hijo unigénito al mundo para que vivamos por él")
print(translation)

kay iñiqmasinchiktam qawachiwarqanchik diospa churinta kay pachaman kachamusqanwan huk runata kuyapayawananchikpaq<END>


In [76]:
translation = translate("como podemos identificar a los verdaderos adoradores de dios")
print(translation)
#¿Imaynatam cheqap cristianokunata yanapachwan?<END>

ñoqanchikpas nichwanmi cheqap yupaychaqkunata dios yupaychaqkunata yupaychaqkunataqa<END>


In [77]:
translation = translate(limpiar_texto_avanzado("Hola mi nombre es Maria y vivo por el rio"))
print(translation)
#Mariawan warmichaymanta wayqay waytaman<END>

diospa sutinmi arriendaqa hinaspa kaypi kachkan<END>


In [78]:
translation = translate("el perro cuida la casa")
print(translation)
#Wasi qura allinta yanu.<END>

wasi wasi qiruy<END>


In [79]:
translation = translate("al perro que cuida la casa")
print(translation)
#Wasi qura allinta yanu.<END>

wasiyoq kaspaykiqa wasitam nispa<END>


In [80]:
translation = translate("al perro se le cría para que cuide la casa")
print(translation)
#Allqutaqa wasi qawanapaqmi.<END>

allqutaqa wasi qawananpaqmi uywanku<END>


In [81]:
translation = translate("para el dolor de muelas es bueno retener en la boca agua con sal")
print(translation)
#Kay nanay simiqa ancha kanchaymi.<END>

kiru nanaypaqqa kachi yakutam amuna<END>


In [82]:
translation = translate("el perro está ladrando al ladrón")
print(translation)
#Allqum suwata anyachkan.<END>

allqum suwata anyachkan<END>


In [83]:
translation = translate("el poder del viento causa miedo")
print(translation)
#Wayrapa allinqa manchakuypaqmi.<END>

wayrapa atiyninqa manchakuypaqmi<END>


In [84]:
translation = translate("vamos a pescar en ese río")
print(translation)
#Wak mayupi pasarqusun.<END>

wak mayupi challwakamusun<END>


In [85]:
translation = translate("entonces tenemos que demostrar nuestro valor con buens acciones")
print(translation)
#Chaynaqa allin yuyayniyoq kanapaqyá kasun allin noticiamanta willakuy allin kasqanta.<END>

chaynaqa chaninchasqanchiktam qawachinanchik allin kaqkunata<END>


In [81]:
import pandas as pd
ruta_csv = 'SentenciasEspv2.csv'

# Cargar el archivo CSV en un DataFrame
dfsentencias = pd.read_csv(ruta_csv,sep='|')
#dfsentencias=dfsentencias.head(500)
# Mostrar las primeras filas
print(dfsentencias.head())


                                             espanol
0             Tocar el xilófono es mi hobby favorito
1  Inmediatamente te envío toda la información a ...
2             ¿Es la violencia innata al ser humano?
3  Comer sopa de tortuga está prohibido, porque e...
4  Este fin de semana voy a ir al cine con mi esposa


In [106]:
dfsentencias.describe

<bound method NDFrame.describe of                                                 espanol
0                Tocar el xilófono es mi hobby favorito
1     Inmediatamente te envío toda la información a ...
2                ¿Es la violencia innata al ser humano?
3     Comer sopa de tortuga está prohibido, porque e...
4     Este fin de semana voy a ir al cine con mi esposa
...                                                 ...
5442  Primero que nada vamos a hacer un ejercicio de...
5443     Ayer llovió mucho, y llegue tarde a mi reunión
5444               El jeroglífico tiene un pez amarillo
5445              Recomiéndame un libro de gastronomía.
5446         ¿Cuánto dinero quieres gastar en la bolsa?

[5447 rows x 1 columns]>

In [82]:
dfsentencias['espanol'] = dfsentencias['espanol'].apply(limpiar_texto_avanzado)

In [85]:
dfsentencias.describe

<bound method NDFrame.describe of                                                 espanol
0                tocar el xilófono es mi hobby favorito
1     inmediatamente te envío toda la información a ...
2                  es la violencia innata al ser humano
3     comer sopa de tortuga está prohibido porque es...
4     este fin de semana voy a ir al cine con mi esposa
...                                                 ...
5442  primero que nada vamos a hacer un ejercicio de...
5443      ayer llovió mucho y llegue tarde a mi reunión
5444               el jeroglífico tiene un pez amarillo
5445               recomiéndame un libro de gastronomía
5446           cuánto dinero quieres gastar en la bolsa

[5447 rows x 1 columns]>

In [87]:
dfsentencias['sentencia_quechua'] = dfsentencias['espanol'].apply(translate)

In [88]:
dfsentencias.describe

<bound method NDFrame.describe of                                                 espanol  \
0                tocar el xilófono es mi hobby favorito   
1     inmediatamente te envío toda la información a ...   
2                  es la violencia innata al ser humano   
3     comer sopa de tortuga está prohibido porque es...   
4     este fin de semana voy a ir al cine con mi esposa   
...                                                 ...   
5442  primero que nada vamos a hacer un ejercicio de...   
5443      ayer llovió mucho y llegue tarde a mi reunión   
5444               el jeroglífico tiene un pez amarillo   
5445               recomiéndame un libro de gastronomía   
5446           cuánto dinero quieres gastar en la bolsa   

                                      sentencia_quechua  
0     chiqap takayniyqa hayka rikchaq karqusqam karq...  
1     hinaptinqa chayllam correginki lliw runakuna c...  
2                            runakuna hamuchkanñam<END>  
3     tutayaq kasqanrayku

In [90]:
#output_excel_path = 'traducciones_quechua_500_30_epocas.xlsx'
ruta_archivo_inferencias = 'traducciones_quechua_5447_50_epocas_dataset_v4_orig_AV.xlsx'
dfsentencias.to_excel(ruta_archivo_inferencias, index=False)

print(f"DataFrame guardado en {ruta_archivo_inferencias}")

DataFrame guardado en traducciones_quechua_5447_50_epocas_dataset_v4_orig_AV.xlsx


##Interfaz web

In [37]:
!pip install flask
!pip install pyngrok


  Using cached flask-3.1.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
Using cached flask-3.1.1-py3-none-any.whl (103 kB)
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
  Using cached pyngrok-7.2.12-py3-none-any.whl.metadata (9.4 kB)
Using cached pyngrok-7.2.12-py3-none-any.whl (26 kB)


In [38]:
!ngrok authtoken 2oY1WqlLxmnJ6HACnG75eB87EoB_5qtkc2t6hRQdn36j4crbB

Authtoken saved to configuration file: /home/jovyan/.config/ngrok/ngrok.yml


In [39]:
import torch
from flask import Flask, render_template_string, request, jsonify
from pyngrok import ngrok

# Carga el modelo de traducción
"""#""model = transformer(eng_batch,
                    kn_batch,
                    encoder_self_attention_mask.to(device),
                    decoder_self_attention_mask.to(device),
                    decoder_cross_attention_mask.to(device),
                    enc_start_token=False,
                    enc_end_token=False,
                    dec_start_token=True,
                    dec_end_token=True)
model_path = './optimizador_transformer_model_3_capas_50epocs122207.pth'
model = torch.load(model_path)
model.eval()  # Configura el modelo para inferencia"""

app = Flask(__name__)

@app.route('/')
def index():
    html_template = '''
    <!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Reconocimiento y Traducción de Voz</title>
        <link rel="stylesheet" href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css">
        <style>
    /* Estilo general del cuerpo de la página */
    body {
        font-family: Arial, sans-serif;
        background: linear-gradient(135deg, #e0e7ff, #e9ecef);
        color: #343a40;
        display: flex;
        justify-content: center;
        align-items: center;
        min-height: 100vh;
        margin: 0;
    }

    /* Contenedor principal */
    .container {
        background-color: #ffffff;
        padding: 2rem;
        border-radius: 12px;
        box-shadow: 0px 4px 16px rgba(0, 0, 0, 0.1);
        max-width: 600px;
        width: 90%;
        text-align: center;
        transition: transform 0.2s ease;
    }

    /* Efecto al hacer hover en el contenedor */
    .container:hover {
        transform: scale(1.02);
        box-shadow: 0px 6px 20px rgba(0, 0, 0, 0.15);
    }

    /* Título principal */
    h1 {
        font-size: 2.5rem;
        margin-bottom: 1rem;
        color: #495057;
        text-transform: uppercase;
        letter-spacing: 2px;
    }

    /* Botón de reconocimiento de voz */
    button {
        font-size: 1.2rem;
        padding: 0.75rem 1.5rem;
        background: linear-gradient(135deg, #007bff, #0056b3);
        color: #fff;
        border: none;
        border-radius: 8px;
        transition: all 0.3s ease;
        box-shadow: 0px 4px 8px rgba(0, 123, 255, 0.3);
        cursor: pointer;
        outline: none;
    }

    /* Efecto hover y activo en el botón */
    button:hover {
        background: linear-gradient(135deg, #0056b3, #004085);
        box-shadow: 0px 6px 12px rgba(0, 85, 179, 0.4);
        transform: translateY(-2px);
    }

    button:active {
        transform: translateY(1px);
    }

    /* Texto del resultado */
    .lead {
        font-size: 1.5rem;
        color: #007bff;
        margin-top: 1.5rem;
        transition: color 0.3s ease;
    }

    /* Texto del resultado cuando se actualiza */
    #result {
        font-weight: bold;
        color: #495057;
        font-size: 1.3rem;
        margin-top: 1rem;
        transition: color 0.3s ease, transform 0.2s ease;
    }

    #result.updated {
        color: #28a745;
        transform: scale(1.05);
    }

    /* Añadir un efecto de subrayado animado al pasar el ratón sobre el título */
    h1::after {
        content: '';
        display: block;
        width: 50px;
        height: 3px;
        background-color: #007bff;
        margin: 8px auto 0;
        transition: width 0.3s ease;
    }

    h1:hover::after {
        width: 100px;
    }
</style>
    </head>
    <body>
        <div class="container mt-5 text-center">
            <h1>Traducción de Voz (español - quechua)</h1>
            <button id="speakButton" class="btn btn-primary mt-3">Hablar</button>
            <div class="mt-4">
                <h2>Resultado:</h2>
                <p id="result" class="lead"></p>
                <h2>Traducción:</h2>
                <p id="translation" class="lead"></p>
            </div>
        </div>
        <script>
            document.getElementById('speakButton').addEventListener('click', function() {
                const recognition = new (window.SpeechRecognition || window.webkitSpeechRecognition)();
                recognition.lang = 'es-ES';
                recognition.start();
                recognition.onresult = async function(event) {
                    const transcript = event.results[0][0].transcript;
                    document.getElementById('result').textContent = transcript;

                    // Envía el texto reconocido al servidor para traducir
                    const response = await fetch('/translate2', {
                        method: 'POST',
                        headers: {
                            'Content-Type': 'application/json',
                        },
                        body: JSON.stringify({ text: transcript }),
                    });
                    const data = await response.json();
                    document.getElementById('translation').textContent = data.translation;
                };
                recognition.onerror = function(event) {
                    console.error('Error:', event.error);
                };
            });
        </script>
    </body>
    </html>
    '''
    return render_template_string(html_template)

transformer.eval()
def translate(eng_sentence):
  eng_sentence = (eng_sentence,)
  kn_sentence = ("",)
  for word_counter in range(max_sequence_length):
    encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask= create_masks(eng_sentence, kn_sentence)
    predictions = transformer(eng_sentence,
                              kn_sentence,
                              encoder_self_attention_mask.to(device),
                              decoder_self_attention_mask.to(device),
                              decoder_cross_attention_mask.to(device),
                              enc_start_token=False,
                              enc_end_token=False,
                              dec_start_token=True,
                              dec_end_token=False)
    next_token_prob_distribution = predictions[0][word_counter]
    next_token_index = torch.argmax(next_token_prob_distribution).item()
    next_token = index_to_target[next_token_index]
    kn_sentence = (kn_sentence[0] + next_token, )
    if next_token == END_TOKEN:
      break
  return kn_sentence[0]

@app.route('/translate2', methods=['POST'])
def translate2():
    data = request.get_json()
    input_text = data.get('text')
    print(input_text)
    # Realiza la inferencia con el modelo de traducción
    #with torch.no_grad():
        # Aquí debes procesar el input_text para que sea compatible con el modelo y convertir la salida del modelo en texto traducido.
    translation = translate(limpiar_texto_avanzado(input_text))
    #print("Check pint", translation)

    return jsonify({'translation': translation.replace('<END>', '')})

if __name__ == '__main__':
    public_url = ngrok.connect(5000)
    print(f"La aplicación está disponible en: {public_url}")
    app.run(port=5000)


La aplicación está disponible en: NgrokTunnel: "https://f8e36405cf91.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [20/Jul/2025 03:41:27] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2025 03:41:28] "GET /favicon.ico HTTP/1.1" 404 -


estoy sintiendo frío


127.0.0.1 - - [20/Jul/2025 03:41:48] "POST /translate2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2025 03:41:48] "POST /translate2 HTTP/1.1" 200 -


estoy sintiendo frío
Dios nos ayudará


127.0.0.1 - - [20/Jul/2025 03:42:34] "POST /translate2 HTTP/1.1" 200 -


Entonces tenemos que demostrar nuestro valor con buenas acciones


127.0.0.1 - - [20/Jul/2025 03:46:06] "POST /translate2 HTTP/1.1" 200 -
